In [ ]:
# ================================================================
# GENERATE SYNTHETIC MEDICAL DATA — Google Colab
# Capstone Project: Healthy Life & Well-being
# DBS Coding Camp x Kampus Merdeka — Data Science Path
#
# Referensi medis:
# [1] ACC/AHA Guidelines — Cardiovascular Risk (2019)
# [2] ADA Standards of Medical Care in Diabetes (2023)
# [3] KDIGO Clinical Practice Guideline for CKD (2022)
# [4] WHO — Haemoglobin concentrations for anaemia diagnosis (2011)
# [5] Fauci et al., Harrison's Principles of Internal Medicine
# [6] Data asli: merge_DocLab_Clean.csv (statistik per kelas)
# ================================================================

# ============================================================
# 1. INSTALL & IMPORT
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# ============================================================
# 2. NILAI RUJUKAN KLINIS (untuk referensi & validasi)
# ============================================================
NILAI_RUJUKAN = {
    'cholesterol total': (0,   200),   # < 200 normal
    'creatinin':         (0.6, 1.1),   # normal range
    'fbs':               (70,  100),   # gula darah puasa
    'rbs':               (70,  110),   # gula darah sewaktu
    'hgb':               (12,  16),    # hemoglobin
    'lymfosit%':         (20,  35),    # limfosit
    'mch':               (27,  34),
    'mchc':              (32,  36),
    'mcv':               (80,  100),
    'ureum':             (17,  43),
    'wbc':               (4,   11),
}

# ============================================================
# 3. PARAMETER DISTRIBUSI PER KELAS
# Format: (mean, std, min_clip, max_clip, desimal)
#
# Basis: statistik data asli merge_DocLab_Clean.csv
#        + literatur medis per kelas
# ============================================================

# ------------------------------------------------------------------
# NM_UNIT = 0 : NORMAL (tidak ada diagnosa dokter)
# Pola: semua parameter mendekati atau dalam rentang rujukan
# Referensi [6]: data asli NM_UNIT=0
# ------------------------------------------------------------------
PARAMS_0 = {
    'cholesterol total': (188,  35,  111, 300,  2),
    'creatinin':         (1.05, 0.30, 0.40, 2.50, 2),
    'fbs':               (105,  22,   70,  200,  1),
    'rbs':               (130,  40,   70,  300,  1),
    'hgb':               (12.5, 1.5,  7.0,  18.0, 1),
    'lymfosit%':         (27.0, 8.0,  10,   55,   1),
    'mch':               (28.2, 2.5,  18,   35,   1),
    'mchc':              (33.2, 1.5,  28,   37,   1),
    'mcv':               (85.0, 7.0,  65,  105,   1),
    'ureum':             (35.0, 15.0, 10,  100,   1),
    'wbc':               (8.5,  2.5,  3.5,  18,   1),
}

# ------------------------------------------------------------------
# NM_UNIT = 1 : JANTUNG (SPES. JANTUNG DAN PEMBULUH DARAH)
# Pola klinis:
#   - Kolesterol TINGGI (dislipidemia) [Ref 1]
#   - FBS/RBS TINGGI (DM sebagai komorbid utama) [Ref 2]
#   - Kreatinin SEDIKIT NAIK (sindrom kardiorenal) [Ref 1]
#   - HGB SEDIKIT RENDAH (anemia pada gagal jantung) [Ref 4]
#   - WBC SEDIKIT NAIK (inflamasi sistemik) [Ref 5]
#   - Ureum SEDIKIT NAIK (kardiorenal) [Ref 1]
# Referensi data asli [6]: mean chol=215, creat=1.4, fbs=151, hgb=11.7
# ------------------------------------------------------------------
PARAMS_1 = {
    'cholesterol total': (215,  38,  130, 340,  2),
    'creatinin':         (1.35, 0.50, 0.50, 5.00, 2),
    'fbs':               (148,  55,   72,  450,  1),
    'rbs':               (195,  85,   72,  600,  1),
    'hgb':               (11.8, 1.80, 7.0,  17.0, 1),
    'lymfosit%':         (22.0, 9.0,  5,    50,   1),
    'mch':               (28.0, 3.00, 18,   36,   1),
    'mchc':              (33.0, 2.00, 27,   37,   1),
    'mcv':               (84.0, 8.00, 60,  105,   1),
    'ureum':             (42.0, 18.0, 10,  150,   1),
    'wbc':               (9.5,  3.00, 3.5,  20,   1),
}

# ------------------------------------------------------------------
# NM_UNIT = 2 : PENYAKIT DALAM / METABOLIK
# Pola klinis:
#   - Kreatinin SANGAT TINGGI (CKD — Chronic Kidney Disease) [Ref 3]
#   - Ureum SANGAT TINGGI (azotemia pada CKD) [Ref 3]
#   - FBS/RBS SANGAT TINGGI (Diabetes Mellitus) [Ref 2]
#   - HGB RENDAH (anemia penyakit kronik) [Ref 4]
#   - Kolesterol bervariasi [Ref 6]
# Referensi data asli [6]: mean creat=3.1, fbs=195, hgb=10.5, ureum=79
# ------------------------------------------------------------------
PARAMS_2 = {
    'cholesterol total': (200,  38,  120, 310,  2),
    'creatinin':         (3.00, 2.50, 0.40, 25.0, 2),
    'fbs':               (188,  80,   72,  500,  1),
    'rbs':               (220, 110,   50,  700,  1),
    'hgb':               (10.5, 1.80, 5.5,  16.0, 1),
    'lymfosit%':         (25.0, 9.0,  5,    55,   1),
    'mch':               (27.5, 3.00, 18,   36,   1),
    'mchc':              (32.0, 2.00, 24,   37,   1),
    'mcv':               (86.0, 7.00, 65,  105,   1),
    'ureum':             (78.0, 45.0, 10,  400,   1),
    'wbc':               (9.2,  3.50, 3.0,  30,   1),
}

# ------------------------------------------------------------------
# NM_UNIT = 3 : PARU-PARU (TB / PPOK / Pneumonia)
# Pola klinis:
#   - HGB RENDAH (anemia pada TB aktif) [Ref 4, 5]
#   - WBC TINGGI (leukositosis pada infeksi paru) [Ref 5]
#   - Lymfosit% RENDAH (lymphopenia pada TB) [Ref 5]
#   - MCH/MCHC/MCV RENDAH (anemia mikrositik) [Ref 4]
#   - Kolesterol RENDAH-NORMAL [Ref 5]
# Referensi data asli [6]: mean hgb=10.5, wbc=10.2, lymfosit=19
# ------------------------------------------------------------------
PARAMS_3 = {
    'cholesterol total': (180,  35,  110, 280,  2),
    'creatinin':         (1.00, 0.40, 0.40, 3.50, 2),
    'fbs':               (115,  35,   72,  300,  1),
    'rbs':               (145,  55,   70,  400,  1),
    'hgb':               (10.8, 1.80, 6.0,  16.0, 1),
    'lymfosit%':         (19.0, 7.00, 5,    40,   1),
    'mch':               (26.5, 3.00, 17,   34,   1),
    'mchc':              (31.5, 2.00, 24,   36,   1),
    'mcv':               (82.0, 8.00, 60,  100,   1),
    'ureum':             (35.0, 15.0, 10,  120,   1),
    'wbc':               (10.5, 3.50, 3.5,  25,   1),
}

# ============================================================
# 4. FUNGSI GENERATE SAMPEL
# ============================================================
def clip_round(val, lo, hi, dec):
    return round(float(np.clip(val, lo, hi)), dec)

def generate_samples(n, params, gender_prob, age_mean, age_std,
                     age_min, age_max, label, missing_rate=0.0):
    """
    Generate n baris data sintetis.

    Parameters
    ----------
    n            : jumlah baris
    params       : dict {kolom: (mean, std, min, max, desimal)}
    gender_prob  : [prob_female, prob_male] → [0, 1]
    age_mean/std : distribusi usia
    age_min/max  : clip usia
    label        : nilai NM_UNIT
    missing_rate : proporsi missing per sel (0.0 = tidak ada missing)
    """
    rows = []
    for _ in range(n):
        gender = int(np.random.choice([0, 1], p=gender_prob))
        age    = int(np.clip(np.random.normal(age_mean, age_std),
                             age_min, age_max))
        row = {'JENIS_KELAMIN': gender, 'UMUR_TAHUN': age, 'NM_UNIT': label}
        for col, (mu, sd, lo, hi, dec) in params.items():
            if missing_rate > 0 and np.random.rand() < missing_rate:
                row[col] = np.nan
            else:
                row[col] = clip_round(np.random.normal(mu, sd), lo, hi, dec)
        rows.append(row)
    return rows

# ============================================================
# 5. GENERATE DATA
# Jumlah per kelas — sesuaikan jika perlu
# Gender prob: [prob_perempuan(0), prob_laki(1)]
# Usia: dari analisis data asli per kelas
# ============================================================
N_NORMAL     = 800   # NM_UNIT = 0
N_JANTUNG    = 600   # NM_UNIT = 1
N_DALAM      = 800   # NM_UNIT = 2
N_PARU       = 400   # NM_UNIT = 3

rows  = generate_samples(N_NORMAL,  PARAMS_0, [0.89, 0.11], 53.7, 17.0, 15, 92, 0)
rows += generate_samples(N_JANTUNG, PARAMS_1, [0.55, 0.45], 60.8, 12.6, 20, 95, 1)
rows += generate_samples(N_DALAM,   PARAMS_2, [0.59, 0.41], 57.3, 13.7, 16, 92, 2)
rows += generate_samples(N_PARU,    PARAMS_3, [0.45, 0.55], 48.0, 14.0, 15, 85, 3)

df_syn = pd.DataFrame(rows)

# Urutan kolom
COL_ORDER = ['JENIS_KELAMIN', 'UMUR_TAHUN',
             'cholesterol total', 'creatinin', 'fbs', 'rbs',
             'hgb', 'lymfosit%', 'mch', 'mchc', 'mcv', 'ureum', 'wbc',
             'NM_UNIT']
df_syn = df_syn[COL_ORDER]

print("✅ Data sintetis berhasil dibuat!")
print(f"   Shape  : {df_syn.shape}")
print(f"   Missing: {df_syn.isnull().sum().sum()} sel")
print("\nDistribusi kelas:")
label_map = {0: 'Normal', 1: 'Jantung', 2: 'Peny.Dalam', 3: 'Paru'}
print(df_syn['NM_UNIT'].map(label_map).value_counts())

# ============================================================
# 6. SIMPAN FILE
# ============================================================
OUTPUT_PATH = 'synthetic_data_clean.csv'
df_syn.to_csv(OUTPUT_PATH, index=False)
print(f"\n💾 Disimpan sebagai: {OUTPUT_PATH}")

# ============================================================
# 7. VALIDASI — STATISTIK PER KELAS
# ============================================================
print("\n" + "="*60)
print("VALIDASI: Mean per kelas vs Nilai Rujukan")
print("="*60)

key_cols = ['cholesterol total', 'creatinin', 'fbs', 'hgb', 'ureum', 'wbc', 'lymfosit%']
mean_per_class = df_syn.groupby('NM_UNIT')[key_cols].mean().round(2)
mean_per_class.index = mean_per_class.index.map(label_map)
display(mean_per_class)

# ============================================================
# 8. VISUALISASI — DISTRIBUSI PER KELAS
# ============================================================
fig, axes = plt.subplots(3, 4, figsize=(18, 13))
axes = axes.flatten()

lab_cols = ['cholesterol total', 'creatinin', 'fbs', 'rbs',
            'hgb', 'lymfosit%', 'mch', 'mchc', 'mcv', 'ureum', 'wbc']
colors   = {0: '#2196F3', 1: '#F44336', 2: '#FF9800', 3: '#4CAF50'}

for i, col in enumerate(lab_cols):
    ax = axes[i]
    for unit, label in label_map.items():
        subset = df_syn[df_syn['NM_UNIT'] == unit][col]
        ax.hist(subset, bins=30, alpha=0.5, label=label,
                color=colors[unit], density=True)
    # Garis nilai rujukan
    if col in NILAI_RUJUKAN:
        lo, hi = NILAI_RUJUKAN[col]
        ax.axvline(lo, color='black', linestyle='--', linewidth=1, alpha=0.6)
        ax.axvline(hi, color='black', linestyle='--', linewidth=1, alpha=0.6)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=7)

# Sembunyikan subplot kosong
for j in range(len(lab_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi Parameter Lab per Kelas\n(garis hitam = nilai rujukan)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('distribusi_synthetic.png', dpi=120, bbox_inches='tight')
plt.show()
print("📊 Plot disimpan: distribusi_synthetic.png")

# ============================================================
# 9. HEATMAP KORELASI
# ============================================================
plt.figure(figsize=(14, 12))
corr = df_syn.select_dtypes(include='number').corr(method='pearson')
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Heatmap Korelasi Pearson — Synthetic Data', fontsize=13)
plt.tight_layout()
plt.savefig('heatmap_synthetic.png', dpi=120, bbox_inches='tight')
plt.show()
print("📊 Plot disimpan: heatmap_synthetic.png")

print("\n✅ Selesai! File siap digunakan untuk modelling.")